# Step 2c: GMACORE-TM pretrainingAdds the two stability mechanisms to GMACORE:- a **momentum encoder** producing the keys, synchronised by EMA (Equation 19)- **age-based weighting** of queued negatives, gamma_i = gamma ** age (Equations 23, 24)Everything else is identical to the GMACORE run: same corpus, same encoder,same seeds, same fixed evaluation subset.## What this run is testingManualAug-CL and GMACORE both collapse on this corpus: effective rank fallsfrom 32 at initialisation to about 2 within 15 epochs, with positive alignmentat 0.99, which is the degenerate solution.The hypothesis is that queue negatives without a momentum encoder are thecause. Stale keys let the encoder reduce its loss by drifting away from itsown past rather than by separating molecules, and drifting is cheapest inside alow-dimensional subspace. A momentum encoder keeps stored keys consistent withthe current representation, which should remove that escape route.**Watch `rank`.** If it holds in the tens, the hypothesis is confirmed and thetwo baselines need in-batch negatives rather than a queue. If it collapses to~2 here as well, the problem lies elsewhere and the momentum encoder is not themissing piece.`gan` and `disc` are expected to read 0.0000 regardless: the discriminatorminimises the same quantity the generator does, so it reaches zero by emittinga constant. That is a separate structural issue and does not affect theencoder-collapse question being tested here.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import os

os.environ["PYTHONWARNINGS"] = "ignore"

import json
import logging
import sys
import time
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)

try:
    import torch_geometric
except ImportError:
    import subprocess

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"], check=True)

try:
    import psutil
except ImportError:
    import subprocess

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psutil"], check=True)
    import psutil

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from tqdm.auto import tqdm

print(f"PyTorch {torch.__version__} | PyG {torch_geometric.__version__}")

## Configuration

In [ ]:
WORKSPACE = Path("/workspace") if Path("/workspace").exists() else Path.cwd()

VARIANT = "gmacore_tm_fixedgan"
GRAPH_DIR = WORKSPACE / "data" / "graphs" / "pubchem"
OUTPUT_DIR = WORKSPACE / "runs" / VARIANT

# ----- encoder (identical to ManualAug-CL) -----
HIDDEN_DIM = 128
OUTPUT_DIM = 128
NUM_LAYERS = 3
DROPOUT = 0.10

TEMPERATURE = 0.07
QUEUE_SIZE = 50_000

# Diagnostic switch. With USE_QUEUE = False the contrastive loss uses in-batch
# negatives only, which is the SimCLR arrangement. Queue negatives are produced
# by earlier encoder states, and without a momentum encoder to keep them
# consistent they are the prime suspect for the dimensional collapse.
# Raise BATCH_SIZE when disabling the queue, since in-batch negatives are then
# the only ones available.
USE_QUEUE = True

BATCH_SIZE = 64
# Reduced from the 5e-4 of Table 3. At 5e-4 the encoder reached the collapsed
# solution within one epoch and could not escape it, since the gradient at
# exact collapse is symmetric and vanishing.
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 150
GRAD_CLIP = 5.0
# The similarity regulariser is DISABLED.
#
# L_sim = ||q/|q| - k/|k|||^2 is globally minimised by mapping every molecule
# to the same direction. It has a trivial optimum, rewards nothing else, and
# is BYOL's objective without BYOL's predictor head or stop-gradient
# asymmetry, which are precisely what stop BYOL collapsing. With it enabled
# the encoder collapsed within one epoch in every run.
#
# Set it back to 0.10 only to reproduce that failure deliberately.
SIMILARITY_WEIGHT = 0.0

# ----- generative augmentation (GMACORE only) -----
GENERATOR_HIDDEN_DIM = 256
NOISE_DIM = 32
DROP_THRESHOLD = 0.25
HARD_DROP = True
GENERATOR_LR = 1e-4
DISCRIMINATOR_LR = 1e-4

# ----- stability mechanisms (GMACORE-TM only) -----
# MOMENTUM: EMA coefficient for the key encoder (Equation 19). A slowly moving
#   key encoder keeps queued negatives consistent with the current
#   representation, which is what makes queue-based negatives well posed.
# DECAY: base coefficient gamma for the age weights (Equation 23). Each stored
#   negative contributes gamma ** age to the partition function.
MOMENTUM = 0.999
DECAY = 0.99

# ----- corrected adversarial objective -----
# The published objective (Equations 12 to 14) has the discriminator MINIMISE
# ||D(G) - D(G')||^2. Its global optimum is a constant output, scoring exactly
# zero, after which the generator receives no gradient. Both the original
# notebook and every run here confirm it: adversarial loss reaching 9.5e-12.
#
# Here the discriminator classifies real against augmented, so a constant
# output scores log 2 = 0.693 rather than 0, and the generator is trained to
# fool it. That is a genuine minimax game with no trivial solution.
#
# A budget is needed as well: a generator that keeps every component fools the
# discriminator trivially, since an unperturbed graph IS real.
# The drop rate is enforced structurally rather than by a soft penalty. A
# penalty on the mean sigmoid score cannot work once the scores saturate:
# the gradient through sigmoid' vanishes and the generator is stuck at
# "keep everything", which is exactly what happened (keep 1.00, drop 0.00).
#
# Instead a fixed fraction of the lowest-scoring components is dropped every
# step, so the augmentation budget is guaranteed and the adversarial game
# decides only WHICH components go.
KEEP_RATIO = 0.75

# ----- runtime -----
SEED = 42
NUM_DATA_WORKERS = 8
CHECKPOINT_EVERY = 10
EMBED_EVERY = 10
EVAL_SUBSET_SIZE = 10_000
EVAL_SUBSET_SEED = 12345   # MUST match the ManualAug-CL run
RESUME = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "embeddings").mkdir(exist_ok=True)

print(f"Variant : {VARIANT}")
print(f"Graphs  : {GRAPH_DIR}")
print(f"Output  : {OUTPUT_DIR}")
print(f"Device  : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

In [ ]:
import random


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True


set_seed(SEED)
print(f"Seed: {SEED}")

## Load the featurized corpus

In [ ]:
MANIFEST_NAME = "manifest.json"


def shard_path(directory, index):
    return os.path.join(str(directory), f"shard_{index:05d}.pt")


def read_manifest(directory):
    with open(os.path.join(str(directory), MANIFEST_NAME), encoding="utf-8") as handle:
        return json.load(handle)


class ShardedGraphDataset(torch.utils.data.Dataset):
    def __init__(self, directory, max_shards=None):
        self.directory = str(directory)
        self.manifest = read_manifest(directory)

        counts = list(self.manifest["shard_counts"])
        if max_shards is not None:
            counts = counts[:max_shards]
        self.num_shards = len(counts)

        self.graphs = []
        for index in tqdm(range(self.num_shards), desc="Loading shards"):
            self.graphs.extend(torch.load(shard_path(self.directory, index), weights_only=False))

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, index):
        return self.graphs[index]


dataset = ShardedGraphDataset(GRAPH_DIR)
NODE_DIM = dataset.manifest["node_dim"]
EDGE_DIM = dataset.manifest["edge_dim"]

print(f"\nGraphs   : {len(dataset):,}")
print(f"node_dim : {NODE_DIM} | edge_dim : {EDGE_DIM}")

In [ ]:
# Same fixed evaluation subset as the baseline, drawn from its own generator so
# it does not depend on training-time RNG state.
eval_rng = np.random.default_rng(EVAL_SUBSET_SEED)
eval_indices = np.sort(eval_rng.choice(
    len(dataset), size=min(EVAL_SUBSET_SIZE, len(dataset)), replace=False
))

eval_loader = DataLoader(
    torch.utils.data.Subset(dataset, eval_indices.tolist()), batch_size=256, shuffle=False
)
np.save(OUTPUT_DIR / "eval_indices.npy", eval_indices)

print(f"Evaluation subset: {len(eval_indices):,} molecules (seed {EVAL_SUBSET_SEED})")
print(f"First 3 indices  : {eval_indices[:3].tolist()}")

# Verify the subset matches the baseline run, since a mismatch would silently
# invalidate every cross-variant comparison.
baseline_indices_path = WORKSPACE / "runs" / "manualaug_cl" / "eval_indices.npy"
if baseline_indices_path.exists():
    baseline_indices = np.load(baseline_indices_path)
    if np.array_equal(baseline_indices, eval_indices):
        print("Matches the ManualAug-CL evaluation subset.")
    else:
        raise RuntimeError(
            "Evaluation subset differs from the ManualAug-CL run. "
            "Check EVAL_SUBSET_SEED, EVAL_SUBSET_SIZE and GRAPH_DIR."
        )
else:
    print("ManualAug-CL subset not found yet; nothing to compare against.")

## Feature statistics

In [ ]:
# Dataset-level feature statistics.
#
# The earlier version standardized x_phys using the statistics of whichever
# molecules happened to share a batch, dividing by (batch_std + 1e-5). When a
# descriptor is near-constant within a batch that divisor is tiny, so the
# feature is amplified by orders of magnitude. The amplification compounds
# through training and drove embedding norms to ~1e12.
#
# Statistics are therefore computed ONCE over a fixed sample of the corpus and
# held constant, so a molecule's features no longer depend on its batch, and
# the standard deviation is floored well away from zero.

_stats_rng = np.random.default_rng(0)
_stats_indices = _stats_rng.choice(len(dataset), size=min(20000, len(dataset)), replace=False)

_cat = torch.cat([dataset[int(i)].x_cat.float() for i in _stats_indices], dim=0)
_phys = torch.cat([dataset[int(i)].x_phys.float() for i in _stats_indices], dim=0)

FEATURE_MEAN = torch.cat([_cat.mean(0), _phys.mean(0)]).to(DEVICE)
FEATURE_STD = torch.cat([_cat.std(0), _phys.std(0)]).to(DEVICE)
FEATURE_STD = torch.clamp(FEATURE_STD, min=1e-2)   # floor, not an epsilon
FEATURE_CLAMP = 10.0

del _cat, _phys

print("Feature statistics (fixed for the whole run):")
print(f"  mean range : [{FEATURE_MEAN.min():.3f}, {FEATURE_MEAN.max():.3f}]")
print(f"  std range  : [{FEATURE_STD.min():.3f}, {FEATURE_STD.max():.3f}]")
print(f"  clamped at : +/- {FEATURE_CLAMP}")

# Masked nodes are replaced by the feature MEAN, which is exactly 0 after
# standardisation. Replacing with raw 0 injects a large constant vector into
# every augmented graph and drives the encoder toward collapse.
MASK_CAT = FEATURE_MEAN[:2].clone()
MASK_PHYS = FEATURE_MEAN[2:].clone()

torch.save(
    {"mean": FEATURE_MEAN.cpu(), "std": FEATURE_STD.cpu(), "clamp": FEATURE_CLAMP},
    OUTPUT_DIR / "feature_stats.pt",
)
print("\nSaved to feature_stats.pt. The other variants must reuse these values.")

## EncoderUnchanged from the baseline. A second instance of the same class serves as thediscriminator: Equation 11 defines it as a separate network, which keeps theadversarial gradient out of the representation used for downstream transfer.

In [ ]:
def normalize_node_features(x_cat, x_phys, eps=1e-5):
    """Concatenate and standardize the node features using fixed statistics.

    Standardization uses corpus-level mean and standard deviation computed once,
    not per-batch statistics. Per-batch standardization makes a molecule's
    representation depend on its batch, and divides by a quantity that can be
    arbitrarily small, which is what caused the embedding explosion.
    """
    x = torch.cat([x_cat.float(), x_phys.float()], dim=-1)
    x = (x - FEATURE_MEAN) / FEATURE_STD
    return torch.clamp(x, -FEATURE_CLAMP, FEATURE_CLAMP)


def mlp(in_dim, hidden_dim, out_dim):
    return nn.Sequential(
        nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, out_dim)
    )


def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.01)


class GraphEncoder(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=128, output_dim=128,
                 num_layers=3, dropout=0.0):
        super().__init__()
        self.node_encoder = mlp(node_dim, hidden_dim, hidden_dim)
        self.edge_encoder = mlp(edge_dim, hidden_dim, hidden_dim)

        self.convs = nn.ModuleList()
        for layer in range(num_layers):
            out_channels = output_dim if layer == num_layers - 1 else hidden_dim
            self.convs.append(GCNConv(hidden_dim, out_channels))

        self.dropout = dropout

        # Both objectives normalise their inputs, so the scale of the pooled
        # representation is a free direction that the optimiser inflates
        # without penalty. Unchecked it reached ~3e4, at which point the
        # contrastive gradient is scaled by 1/|q| ~ 3e-5 and learning stops.
        # LayerNorm pins the scale and keeps gradients well conditioned.
        self.pool_norm = nn.LayerNorm(output_dim)
        self.projection = mlp(output_dim, hidden_dim, output_dim)
        self.apply(init_weights)

    def encode(self, data):
        x = normalize_node_features(data.x_cat, data.x_phys)
        x = self.node_encoder(x)
        _ = self.edge_encoder(data.edge_attr.float())

        # Edge weights, when present, let the generator attenuate individual
        # bonds differentiably. GCNConv accepts a scalar weight per edge; the
        # encoded edge FEATURES are not consumed by GCNConv at all, which is
        # why scaling edge_attr alone had no effect on the output.
        edge_weight = getattr(data, "edge_weight", None)

        for index, conv in enumerate(self.convs):
            x = conv(x, data.edge_index, edge_weight)
            if index < len(self.convs) - 1:
                x = F.relu(x)
                if self.dropout > 0:
                    x = F.dropout(x, p=self.dropout, training=self.training)

        batch = getattr(data, "batch", None)
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        return self.pool_norm(global_mean_pool(x, batch))

    def forward(self, data):
        return self.projection(self.encode(data))


print("Encoder defined.")

## GeneratorNode and edge importance scores are conditioned on a latent noise vector`z_noise ~ N(0, I)` (Equation 9), sampled once per graph and broadcast acrossthat graph's nodes and edges so the perturbation stays coherent within amolecule while remaining stochastic across the batch.Components scoring below `DROP_THRESHOLD` are dropped. Dropping is a discretedecision and not differentiable, so a straight-through estimator applies thehard mask in the forward pass while letting gradients reach the scoring MLPs inthe backward pass. Without it the generator could not be trained at all.

In [ ]:
class _StraightThroughThreshold(torch.autograd.Function):
    """Hard threshold forward, identity gradient backward."""

    @staticmethod
    def forward(ctx, scores, threshold):
        # threshold may be a python float or a 0-dim tensor (a batch quantile).
        return (scores >= threshold).to(scores.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None


def straight_through_threshold(scores, threshold):
    return _StraightThroughThreshold.apply(scores, threshold)


class GraphGenerator(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=256, noise_dim=32,
                 num_layers=3, drop_threshold=0.25, hard_drop=True, keep_ratio=0.75):
        super().__init__()
        self.noise_dim = noise_dim
        self.drop_threshold = drop_threshold
        self.hard_drop = hard_drop
        self.keep_ratio = keep_ratio

        self.node_encoder = mlp(node_dim, hidden_dim, hidden_dim)
        self.edge_encoder = mlp(edge_dim, hidden_dim, hidden_dim)
        self.convs = nn.ModuleList([GCNConv(hidden_dim, hidden_dim) for _ in range(num_layers)])

        self.node_scorer = nn.Sequential(
            nn.Linear(hidden_dim + noise_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.edge_scorer = nn.Sequential(
            nn.Linear(2 * hidden_dim + noise_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.apply(init_weights)

    def _sample_noise(self, data, device):
        batch = getattr(data, "batch", None)
        if batch is None:
            batch = torch.zeros(data.x_cat.size(0), dtype=torch.long, device=device)
        num_graphs = int(batch.max().item()) + 1 if batch.numel() else 1
        return torch.randn(num_graphs, self.noise_dim, device=device)[batch]

    def forward(self, data):
        device = data.x_cat.device

        x = normalize_node_features(data.x_cat, data.x_phys)
        x = self.node_encoder(x)
        _ = self.edge_encoder(data.edge_attr.float())

        for index, conv in enumerate(self.convs):
            x = conv(x, data.edge_index)
            if index < len(self.convs) - 1:
                x = F.relu(x)

        noise = self._sample_noise(data, device)
        node_scores = torch.sigmoid(self.node_scorer(torch.cat([x, noise], dim=-1)))

        source, target = data.edge_index[0], data.edge_index[1]
        edge_scores = torch.sigmoid(
            self.edge_scorer(torch.cat([x[source], x[target], noise[source]], dim=-1))
        )
        return node_scores, edge_scores

    def perturb(self, data, node_scores, edge_scores, keep_ratio=None):
        """Drop the lowest-scoring fraction of nodes and edges.

        The threshold is the (1 - keep_ratio) quantile of the scores in this
        batch, so the fraction dropped is fixed regardless of how the scores
        are distributed. A soft penalty on the mean score cannot achieve this:
        once the sigmoid saturates its gradient vanishes and the generator
        settles on keeping everything.

        Masked nodes take the neutral feature value (the corpus mean, which is
        0 after standardisation) rather than raw zero. Dropped edges are
        attenuated through edge_weight, which GCNConv consumes, so the edge
        path actually influences the output and stays differentiable.
        """
        keep_ratio = self.keep_ratio if keep_ratio is None else keep_ratio

        node_threshold = torch.quantile(node_scores.detach(), 1.0 - keep_ratio)
        edge_threshold = torch.quantile(edge_scores.detach(), 1.0 - keep_ratio)

        node_weight = node_scores * straight_through_threshold(node_scores, node_threshold)
        edge_weight = edge_scores * straight_through_threshold(edge_scores, edge_threshold)

        mask_cat = MASK_CAT.to(data.x_cat.device)
        mask_phys = MASK_PHYS.to(data.x_phys.device)

        out = Data(
            x_cat=node_weight * data.x_cat.float() + (1 - node_weight) * mask_cat,
            x_phys=node_weight * data.x_phys.float() + (1 - node_weight) * mask_phys,
            edge_index=data.edge_index,
            edge_attr=data.edge_attr.float(),
        )
        out.edge_weight = edge_weight.squeeze(-1)
        out.batch = getattr(data, "batch", None)
        out.num_nodes = data.x_cat.size(0)
        return out


print("Generator defined.")

## Queue and objectivesThe adversarial loss (Equation 12) is the squared distance between thediscriminator's representations of the original and augmented graphs. Note thatEquations 13 and 14 have generator and discriminator descending the *same*quantity, so this is an alignment objective rather than a zero-sum game. That isdeliberate: the generator is rewarded for augmentations the discriminator cannotdistinguish from the original, which is what makes the augmented view usable asa contrastive *positive*.

In [ ]:
class MemoryQueue(nn.Module):
    """FIFO queue of L2-normalized negative keys, with age tracking."""

    def __init__(self, size, dim, decay=0.99):
        super().__init__()
        self.size = size
        self.decay = decay
        self.register_buffer("keys", F.normalize(torch.randn(size, dim), dim=1))
        self.register_buffer("age", torch.zeros(size))
        self.register_buffer("valid", torch.zeros(size, dtype=torch.bool))
        self.register_buffer("pointer", torch.zeros(1, dtype=torch.long))

    @property
    def num_valid(self):
        return int(self.valid.sum().item())

    @torch.no_grad()
    def enqueue(self, keys):
        keys = F.normalize(keys.detach(), dim=1)
        batch_size = keys.size(0)
        if batch_size == 0:
            return

        self.age += 1.0
        pointer = int(self.pointer.item())
        indices = (torch.arange(batch_size, device=keys.device) + pointer) % self.size

        self.keys[indices] = keys
        self.age[indices] = 0.0
        self.valid[indices] = True
        self.pointer[0] = (pointer + batch_size) % self.size

    def negatives(self):
        return self.keys[self.valid]

    def decay_weights(self):
        """Age weights gamma_i = gamma ** age_i for the valid entries."""
        gamma = torch.tensor(self.decay, device=self.age.device, dtype=self.age.dtype)
        return torch.pow(gamma, self.age)[self.valid]

    def mean_age(self):
        return float(self.age[self.valid].mean().item()) if self.num_valid else 0.0


def info_nce_loss(query, positive_key, negatives, temperature,
                  in_batch_keys=None, queue_weights=None):
    """Queue-based InfoNCE with in-batch negatives and age-weighted queue.

    Equation 24 weights the EXPONENTIATED similarity of each stored negative:

        gamma_i * exp(s_i / tau) == exp(s_i / tau + log gamma_i)

    so in log-space the weight is an additive term on the logit. Scaling the
    similarity itself instead is a different and incorrect objective: it pulls
    aged negatives toward orthogonality rather than reducing their influence on
    the partition function.

    Age weights apply to queued negatives only. In-batch negatives were
    produced by the current encoder at this step, so their age is zero.
    """
    query = F.normalize(query, dim=1)
    positive_key = F.normalize(positive_key.detach(), dim=1)
    negatives = F.normalize(negatives.detach(), dim=1)

    # Each part is scaled by the temperature here, so that the additive
    # log-weight below is applied after scaling rather than before it.
    logits_pos = (query * positive_key).sum(dim=1, keepdim=True) / temperature
    logits_queue = (query @ negatives.t()) / temperature

    if queue_weights is not None:
        logits_queue = logits_queue + torch.log(
            queue_weights.detach().clamp_min(1e-12)
        ).unsqueeze(0)

    parts = [logits_pos, logits_queue]

    if in_batch_keys is not None:
        in_batch = F.normalize(in_batch_keys.detach(), dim=1)
        logits_in_batch = (query @ in_batch.t()) / temperature
        eye = torch.eye(logits_in_batch.size(0), dtype=torch.bool, device=query.device)
        parts.append(logits_in_batch.masked_fill(eye, float("-inf")))

    logits = torch.cat(parts, dim=1)
    labels = torch.zeros(logits.size(0), dtype=torch.long, device=query.device)
    return F.cross_entropy(logits, labels)


def discriminator_bce(real_logits, augmented_logits):
    """Real against augmented classification (discriminator objective)."""
    return 0.5 * (
        F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits))
        + F.binary_cross_entropy_with_logits(
            augmented_logits, torch.zeros_like(augmented_logits)
        )
    )


def generator_bce(augmented_logits):
    """Non-saturating generator objective: make augmented graphs look real."""
    return F.binary_cross_entropy_with_logits(
        augmented_logits, torch.ones_like(augmented_logits)
    )


# The soft budget penalty was removed: see GraphGenerator.perturb for why a
# quantile threshold replaced it.


print("Queue and objectives defined.")

## Build the models

In [ ]:
set_seed(SEED)

encoder = GraphEncoder(
    node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
).to(DEVICE)

generator = GraphGenerator(
    node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=GENERATOR_HIDDEN_DIM,
    noise_dim=NOISE_DIM, drop_threshold=DROP_THRESHOLD, hard_drop=HARD_DROP,
    keep_ratio=KEEP_RATIO,
).to(DEVICE)

discriminator = GraphEncoder(
    node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM, num_layers=NUM_LAYERS,
).to(DEVICE)

# Momentum encoder: an EMA copy of the query encoder, never updated by
# backpropagation (Equation 19).
import copy

momentum_encoder = copy.deepcopy(encoder).to(DEVICE)
for parameter in momentum_encoder.parameters():
    parameter.requires_grad = False


@torch.no_grad()
def momentum_update():
    for query_param, key_param in zip(encoder.parameters(), momentum_encoder.parameters()):
        key_param.data.mul_(MOMENTUM).add_(query_param.data, alpha=1.0 - MOMENTUM)
    for query_buffer, key_buffer in zip(encoder.buffers(), momentum_encoder.buffers()):
        key_buffer.data.copy_(query_buffer.data)


# Classification head turning the discriminator's graph embedding into a
# single real/augmented logit.
discriminator_head = nn.Linear(OUTPUT_DIM, 1).to(DEVICE)

queue = MemoryQueue(QUEUE_SIZE, OUTPUT_DIM, decay=DECAY).to(DEVICE)

optimizer_encoder = torch.optim.Adam(
    encoder.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
optimizer_generator = torch.optim.Adam(generator.parameters(), lr=GENERATOR_LR)
optimizer_discriminator = torch.optim.Adam(
    list(discriminator.parameters()) + list(discriminator_head.parameters()),
    lr=DISCRIMINATOR_LR,
)


def count(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


num_parameters = count(encoder) + count(generator) + count(discriminator)
print(f"Encoder       : {count(encoder):,}")
print(f"Generator     : {count(generator):,}")
print(f"Discriminator : {count(discriminator):,}")
print(f"Total         : {num_parameters:,}")

train_loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_DATA_WORKERS, pin_memory=(DEVICE == "cuda"),
    drop_last=True, persistent_workers=NUM_DATA_WORKERS > 0,
)
print(f"Steps per epoch: {len(train_loader):,}")

## Training stepOrder per optimization step:1. discriminator update on a detached augmented view (Equation 14)2. encoder update on the contrastive objective3. generator update through the frozen discriminator (Equation 13)4. FIFO enqueue of the new keys, after the encoder step so a key's recorded age   counts the encoder updates elapsed since it was producedThe generator's parameters do not change between steps 1 and 2, so a singleno-grad score computation is shared by both. Step 3 needs its own forward passbecause it requires gradients.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader):
    was_training = model.training
    model.eval()
    chunks = [model.encode(batch.to(DEVICE)).cpu() for batch in loader]
    if was_training:
        model.train()
    return torch.cat(chunks, dim=0)


def train_epoch(epoch):
    encoder.train()
    generator.train()
    discriminator.train()

    totals = {"contrastive": 0.0, "similarity": 0.0, "adversarial": 0.0,
              "discriminator": 0.0, "keep_node": 0.0, "keep_edge": 0.0,
              "dropped_frac": 0.0}
    num_batches = 0
    positive_cosine_sum = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for batch in progress:
        batch = batch.to(DEVICE, non_blocking=True)

        # Scores used by steps 1 and 2. The generator is not updated until
        # step 3, so one sample serves both and saves a forward pass.
        with torch.no_grad():
            node_scores, edge_scores = generator(batch)
            augmented = generator.perturb(batch, node_scores, edge_scores)

        # --- 1. discriminator ---
        optimizer_discriminator.zero_grad(set_to_none=True)
        loss_discriminator = discriminator_bce(
            discriminator_head(discriminator(batch)),
            discriminator_head(discriminator(augmented)),
        )
        loss_discriminator.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(
                list(discriminator.parameters()) + list(discriminator_head.parameters()),
                GRAD_CLIP,
            )
        optimizer_discriminator.step()

        # --- 2. encoder ---
        # A second, independently sampled generated view serves as the key, so
        # both sides of the positive pair are augmented.
        with torch.no_grad():
            key_node_scores, key_edge_scores = generator(batch)
            key_view = generator.perturb(batch, key_node_scores, key_edge_scores)

        query = encoder(augmented)

        # No momentum encoder in GMACORE: keys come from the query encoder.
        # Evaluation mode disables dropout, so the key is a deterministic
        # function of the input and also serves as the similarity reference.
        # Keys come from the momentum encoder, not the query encoder.
        with torch.no_grad():
            momentum_encoder.eval()
            keys = momentum_encoder(key_view)

        negatives = queue.negatives() if USE_QUEUE else keys.new_zeros((0, keys.size(1)))
        if USE_QUEUE and negatives.size(0) == 0:
            negatives = keys.detach()

        queue_weights = queue.decay_weights() if (USE_QUEUE and queue.num_valid) else None
        loss_contrastive = info_nce_loss(
            query, keys, negatives, TEMPERATURE,
            in_batch_keys=keys, queue_weights=queue_weights,
        )
        # Computed on L2-normalized embeddings. On raw embeddings this term
        # is unbounded, and because the contrastive loss normalizes its
        # inputs nothing else constrains the embedding magnitude, so the
        # two objectives fought and the norm diverged.
        loss_similarity = F.mse_loss(
            F.normalize(query, dim=1), F.normalize(keys.detach(), dim=1)
        )
        loss_encoder = loss_contrastive + SIMILARITY_WEIGHT * loss_similarity

        optimizer_encoder.zero_grad(set_to_none=True)
        loss_encoder.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(encoder.parameters(), GRAD_CLIP)
        optimizer_encoder.step()

        # EMA synchronisation, after the encoder step.
        momentum_update()

        # --- 3. generator, through the now-frozen discriminator ---
        optimizer_generator.zero_grad(set_to_none=True)
        gen_node_scores, gen_edge_scores = generator(batch)
        gen_augmented = generator.perturb(batch, gen_node_scores, gen_edge_scores)
        # No budget term: the drop rate is enforced by the quantile threshold,
        # so the generator optimises only which components to drop.
        loss_adversarial = generator_bce(
            discriminator_head(discriminator(gen_augmented))
        )
        loss_adversarial.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(generator.parameters(), GRAD_CLIP)
        optimizer_generator.step()

        # --- 4. enqueue ---
        queue.enqueue(keys)

        with torch.no_grad():
            positive_cosine_sum += float(
                F.cosine_similarity(F.normalize(query, dim=1),
                                    F.normalize(keys, dim=1)).mean()
            )
        totals["contrastive"] += float(loss_contrastive.item())
        totals["similarity"] += float(loss_similarity.item())
        totals["adversarial"] += float(loss_adversarial.item())
        with torch.no_grad():
            totals["keep_node"] += float(gen_node_scores.mean())
            totals["keep_edge"] += float(gen_edge_scores.mean())
            threshold = torch.quantile(gen_node_scores.detach(), 1.0 - KEEP_RATIO)
            totals["dropped_frac"] += float(
                (gen_node_scores < threshold).float().mean()
            )
        totals["discriminator"] += float(loss_discriminator.item())
        num_batches += 1
        progress.set_postfix(
            cl=f"{totals['contrastive'] / num_batches:.4f}",
            gan=f"{totals['adversarial'] / num_batches:.4f}",
        )

    metrics = {key: value / max(num_batches, 1) for key, value in totals.items()}
    metrics["positive_cosine"] = positive_cosine_sum / max(num_batches, 1)
    metrics["queue_mean_age"] = queue.mean_age()
    metrics["queue_occupancy"] = queue.num_valid
    return metrics


def save_checkpoint(path, epoch, metrics):
    torch.save({
        "epoch": epoch,
        "variant": VARIANT,
        "encoder_state_dict": encoder.state_dict(),
        "momentum_encoder_state_dict": momentum_encoder.state_dict(),
        "generator_state_dict": generator.state_dict(),
        "discriminator_state_dict": discriminator.state_dict(),
        "discriminator_head_state_dict": discriminator_head.state_dict(),
        "queue_state_dict": queue.state_dict(),
        "optimizer_encoder": optimizer_encoder.state_dict(),
        "optimizer_generator": optimizer_generator.state_dict(),
        "optimizer_discriminator": optimizer_discriminator.state_dict(),
        "metrics": metrics,
        "config": RUN_CONFIG,
    }, path)


print("Routines defined.")

In [ ]:
RUN_CONFIG = {
    "variant": VARIANT,
    "node_dim": NODE_DIM, "edge_dim": EDGE_DIM,
    "hidden_dim": HIDDEN_DIM, "output_dim": OUTPUT_DIM,
    "num_layers": NUM_LAYERS, "dropout": DROPOUT,
    "temperature": TEMPERATURE, "queue_size": QUEUE_SIZE, "use_queue": USE_QUEUE,
    "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY, "epochs": EPOCHS,
    "grad_clip": GRAD_CLIP, "similarity_weight": SIMILARITY_WEIGHT,
    "generator_hidden_dim": GENERATOR_HIDDEN_DIM, "noise_dim": NOISE_DIM,
    "drop_threshold": DROP_THRESHOLD, "hard_drop": HARD_DROP,
    "generator_lr": GENERATOR_LR, "discriminator_lr": DISCRIMINATOR_LR,
    # Recorded explicitly so the ablation is unambiguous in the saved config.
    "momentum_encoder": True,
    "age_based_weighting": True,
    "momentum": MOMENTUM, "decay": DECAY,
    "adversarial_objective": "real_vs_augmented_bce",
    "keep_ratio": KEEP_RATIO,
    "seed": SEED, "eval_subset_seed": EVAL_SUBSET_SEED,
    "num_parameters": num_parameters,
    "corpus": {
        "graph_dir": str(GRAPH_DIR),
        "num_graphs": len(dataset),
        "manifest": dataset.manifest,
    },
}

with open(OUTPUT_DIR / "run_config.json", "w", encoding="utf-8") as handle:
    json.dump(RUN_CONFIG, handle, indent=2)

print(json.dumps({k: v for k, v in RUN_CONFIG.items() if k != "corpus"}, indent=2))

## Resume

In [ ]:
start_epoch = 1
history = []
resource_log = []
best_loss = float("inf")

latest = sorted(OUTPUT_DIR.glob("checkpoint_epoch_*.pt"))
if RESUME and latest:
    checkpoint = torch.load(latest[-1], map_location=DEVICE, weights_only=False)
    encoder.load_state_dict(checkpoint["encoder_state_dict"])
    momentum_encoder.load_state_dict(checkpoint["momentum_encoder_state_dict"])
    generator.load_state_dict(checkpoint["generator_state_dict"])
    discriminator.load_state_dict(checkpoint["discriminator_state_dict"])
    discriminator_head.load_state_dict(checkpoint["discriminator_head_state_dict"])
    queue.load_state_dict(checkpoint["queue_state_dict"])
    optimizer_encoder.load_state_dict(checkpoint["optimizer_encoder"])
    optimizer_generator.load_state_dict(checkpoint["optimizer_generator"])
    optimizer_discriminator.load_state_dict(checkpoint["optimizer_discriminator"])
    start_epoch = checkpoint["epoch"] + 1

    for name, store in [("training_history.json", history), ("resource_usage.json", resource_log)]:
        path = OUTPUT_DIR / name
        if path.exists():
            with open(path, encoding="utf-8") as handle:
                store.extend(json.load(handle))

    if history:
        best_loss = min(record["contrastive"] for record in history)
    print(f"Resumed from {latest[-1].name}, continuing at epoch {start_epoch}")
else:
    print("Starting from scratch.")

## Train

In [ ]:
process = psutil.Process()

if start_epoch == 1:
    torch.save(
        extract_embeddings(encoder, eval_loader),
        OUTPUT_DIR / "embeddings" / "embeddings_epoch_000.pt",
    )
    print("Saved epoch-0 embeddings.")

run_start = time.perf_counter()

for epoch in range(start_epoch, EPOCHS + 1):
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    epoch_start = time.perf_counter()
    metrics = train_epoch(epoch)
    epoch_time = time.perf_counter() - epoch_start

    usage = {
        "epoch": epoch,
        "epoch_seconds": round(epoch_time, 2),
        "molecules_per_second": round(len(dataset) / epoch_time, 1),
        "host_rss_gb": round(process.memory_info().rss / 1024**3, 2),
    }
    if DEVICE == "cuda":
        usage["gpu_peak_allocated_gb"] = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
        usage["gpu_peak_reserved_gb"] = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    # Separate the two quantities the contrastive loss confounds.
    #
    # positive_cosine : alignment of the two augmented views of one molecule.
    #                   Should RISE. Falling means the encoder is degrading.
    # queue_cosine    : similarity of current embeddings to queued negatives.
    #                   Rising means negatives are getting harder, which makes
    #                   the loss rise for a healthy reason.
    # effective_rank  : exp(entropy of the singular value spectrum). Near 1 is
    #                   dimensional collapse; the tens is healthy.
    # centred_cosine  : pairwise cosine after removing the population mean, so
    #                   a shared constant offset does not inflate it.
    with torch.no_grad():
        probe = extract_embeddings(encoder, eval_loader)[:512]
        probe_n = F.normalize(probe, dim=1)

        cosine = probe_n @ probe_n.t()
        offdiag = cosine[~torch.eye(cosine.size(0), dtype=torch.bool)]
        metrics["embedding_norm"] = float(probe.norm(dim=1).mean())
        metrics["mean_pairwise_cosine"] = float(offdiag.mean())

        centred = F.normalize(probe - probe.mean(0, keepdim=True), dim=1)
        cc = centred @ centred.t()
        metrics["centred_cosine"] = float(cc[~torch.eye(cc.size(0), dtype=torch.bool)].mean())

        singular = torch.linalg.svdvals(probe - probe.mean(0, keepdim=True))
        spectrum = singular / singular.sum().clamp_min(1e-12)
        spectrum = spectrum[spectrum > 0]
        metrics["effective_rank"] = float(torch.exp(-(spectrum * spectrum.log()).sum()))

        metrics["queue_cosine"] = float(
            (F.normalize(probe, dim=1).to(queue.keys.device) @ queue.negatives().t()).mean()
        ) if queue.num_valid else 0.0

    record = {"epoch": epoch, **metrics}
    history.append(record)
    resource_log.append(usage)

    print(
        f"Epoch {epoch:3d} | cl {metrics['contrastive']:.4f} "
        f"| gan {metrics['adversarial']:.3f} | disc {metrics['discriminator']:.3f} "
        f"| keep {metrics['keep_node']:.2f} | drop {metrics['dropped_frac']:.2f} "
        f"| pos {metrics['positive_cosine']:+.3f} "
        f"| qcos {metrics['queue_cosine']:+.3f} "
        f"| rank {metrics['effective_rank']:5.1f} "
        f"| ccos {metrics['centred_cosine']:+.3f} | {epoch_time:.0f}s"
        + (f" | GPU {usage['gpu_peak_allocated_gb']:.2f} GB" if DEVICE == "cuda" else "")
    )

    if epoch % EMBED_EVERY == 0 or epoch == EPOCHS:
        torch.save(
            extract_embeddings(encoder, eval_loader),
            OUTPUT_DIR / "embeddings" / f"embeddings_epoch_{epoch:03d}.pt",
        )

    if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS:
        save_checkpoint(OUTPUT_DIR / f"checkpoint_epoch_{epoch:03d}.pt", epoch, record)

    if metrics["contrastive"] < best_loss:
        best_loss = metrics["contrastive"]
        save_checkpoint(OUTPUT_DIR / "best_model.pt", epoch, record)

    with open(OUTPUT_DIR / "training_history.json", "w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    with open(OUTPUT_DIR / "resource_usage.json", "w", encoding="utf-8") as handle:
        json.dump(resource_log, handle, indent=2)

save_checkpoint(OUTPUT_DIR / "final_model.pt", EPOCHS, history[-1] if history else {})
print(f"\nComplete in {(time.perf_counter() - run_start) / 3600:.2f} h. "
      f"Best contrastive loss {best_loss:.4f}")

## Summary and comparison with the baseline

In [ ]:
epoch_times = [record["epoch_seconds"] for record in resource_log]
print(f"Variant            : {VARIANT}")
print(f"Epochs completed   : {len(history)}")
print(f"Parameters         : {num_parameters:,}")

if epoch_times:
    print(f"\nMean epoch time    : {np.mean(epoch_times):.1f}s")
    print(f"Total train time   : {np.sum(epoch_times) / 3600:.2f} h")

if DEVICE == "cuda":
    print(f"Peak GPU allocated : {max(r.get('gpu_peak_allocated_gb', 0) for r in resource_log):.2f} GB")
print(f"Peak host RSS      : {max(r['host_rss_gb'] for r in resource_log):.2f} GB")

if history:
    print(f"\nContrastive loss   : {history[0]['contrastive']:.4f} -> {history[-1]['contrastive']:.4f}")
    print(f"Adversarial loss   : {history[0]['adversarial']:.4f} -> {history[-1]['adversarial']:.4f}")

# Side-by-side with the baseline, if it has been run.
baseline_history = WORKSPACE / "runs" / "manualaug_cl" / "training_history.json"
if baseline_history.exists():
    with open(baseline_history, encoding="utf-8") as handle:
        baseline = json.load(handle)
    shared = min(len(baseline), len(history))
    if shared:
        print(f"\nContrastive loss at epoch {shared}:")
        print(f"  ManualAug-CL : {baseline[shared - 1]['contrastive']:.4f}")
        print(f"  GMACORE      : {history[shared - 1]['contrastive']:.4f}")
        print("\nThe manuscript expects GMACORE below the baseline. If it is not,")
        print("inspect the adversarial loss before running GMACORE-TM.")

## NextGMACORE-TM adds the momentum encoder and the age-based weighting of queuednegatives to this configuration. Keep `GRAPH_DIR`, `SEED` and`EVAL_SUBSET_SEED` unchanged for that run as well.